In [0]:
library(dataiku)
library(rpart)
library(dplyr)
library(caret)
library(pROC) # For AUC calculation
library(data.table)
library(mlflow)
library(reticulate)
library(Matrix)
library(purrr) # useful for code optimization
library(themis)
library(doMC)

In [0]:
# Recipe inputs
base_train <- dkuReadDataset("base_train", samplingMethod="head", nbRows=100000)
base_test <- dkuReadDataset("base_test", samplingMethod="head", nbRows=100000)
base_validation <- dkuReadDataset("base_validation", samplingMethod="head", nbRows=100000)

In [0]:
# Combining train and validation datasets to one
# Because we are going to use CV to train the models later
# naming it df_base_train to remain consistent with df naming
df_base_train  <- rbind(base_train, base_validation)

cat("number of rows in combined train data:", nrow(df_base_train), sep = " ")

In [0]:
# setting seed for reproducibility
set.seed(1234)

tune_grid <- expand.grid(
  nrounds = c(47,50, 60,70), # early stopping does not work, we still need to specify nrounds
  max_depth = c(2, 3, 4, 6),
  eta = c(0.09, 0.1, 0.11, 0.12),
  gamma = c(0, 1, 2, 3, 4),
  colsample_bytree = c(0.9, 1.0, 1.1),
  min_child_weight = c(2, 3, 4),
  subsample = c(0.5, 0.6, 0.7, 0.8)
)


# Set up train control with 10-fold cross-validation
train_control <- trainControl(
  method = "cv",
  number = 7,
  classProbs = TRUE,  # Needed for AUC calculation
  summaryFunction = twoClassSummary,
  sampling = "smote", # caret automatically identifies minority class
  search = "random" # random selection of the expanded grid
)

# Detect and register the number of available cores (use all but one)
num_cores <- parallel::detectCores() - 2
registerDoMC(cores = num_cores)  # Enable parallel processing


# Measure the time for a code block to run
system.time({
    # Train the model using grid search with 10-fold CV

    xgb_model <- train(
      damage_binary_2 ~ track_min_dist +
        wind_max_pred +
        rain_total_pred +
        roof_strong_wall_strong +
        roof_strong_wall_light +
        roof_strong_wall_salv +
        roof_light_wall_strong +
        roof_light_wall_light +
        roof_light_wall_salv +
        roof_salv_wall_strong +
        roof_salv_wall_light +
        roof_salv_wall_salv +
        #ls_risk_pct + # Not local variable
        #ss_risk_pct + # Not local variable
        wind_blue_ss +
        wind_yellow_ss +
        wind_orange_ss +
        wind_red_ss +
        rain_blue_ss +
        rain_yellow_ss +
        rain_orange_ss +
        rain_red_ss,
        data = df_base_train2,
        method = "xgbTree",
        trControl = train_control,
        tuneGrid = tune_grid,
        metric = "ROC" # "xgbTree" does not support other metrics for classification tasks (e.g., Kappa or F1)
    )
    Sys.sleep(2)  # This is just an example to simulate a delay
})

# Print best parameters
print(xgb_model$bestTune)

In [0]:
# Extract the best parameters (remove AUC column)
best_params_model <- xgb_model$bestTune

damage_fit_class_full <- train(
          damage_binary_2 ~ wind_max_pred +
            rain_total_pred +
            roof_strong_wall_strong +
            roof_strong_wall_light +
            roof_strong_wall_salv +
            roof_light_wall_strong +
            roof_light_wall_light +
            roof_light_wall_salv +
            roof_salv_wall_strong +
            roof_salv_wall_light +
            roof_salv_wall_salv +
            #ls_risk_pct + # Not local variables
            #ss_risk_pct +
            wind_blue_ss +
            wind_yellow_ss +
            wind_orange_ss +
            wind_red_ss +
            rain_blue_ss +
            rain_yellow_ss +
            rain_orange_ss +
            rain_red_ss +
           track_min_dist, # Confounder adjustment
          data = df_base_train2, # USE TRAINING AND VALIDATION SETS COMBINED
          method = "xgbTree", # XGBoost method
          trControl = trainControl(method = "none"),  # No automatic validation
          tuneGrid = best_params_model # USE BEST PARAMETER
        )

In [0]:
# Sanity Check
# testing on the training datasets (training + validation)

## Outcome prediction on the final_training_df dataset
## default function predict returns class probabilities (has two columns)
y_pred <- predict(damage_fit_class_full,
                  newdata = df_base_train2)


levels(y_pred)


# using table function
conf_matrix <- confusionMatrix(y_pred,
                     df_base_train2$damage_binary_2, # remember to use damage_binary_2
                     positive = "Damage_above_10"
                     )
conf_matrix


accuracy <- conf_matrix$overall['Accuracy']

cat("test-set accuracy of minimal SCM model:", accuracy, sep = " ")

In [0]:
# Logging the model and parameter using MLflow

# set tracking URI
mlflow_set_tracking_uri("http://127.0.0.1:5000")

# Ensure any active run is ended
suppressWarnings(try(mlflow_end_run(), silent = TRUE))

# set experiment
# Logging metrics for model training and the parameters used
mlflow_set_experiment(experiment_name = "U-SCM - XGBOOST classification - CV (Training metircs)")

# Ensure that MLflow has only one run. Start MLflow run once.
run_name <- paste("XGBoost Run", Sys.time())  # Unique name using current time


# Start MLflow run
mlflow_start_run(nested = FALSE)

# Ensure the run ends even if an error occurs
#on.exit(mlflow_end_run(), add = TRUE)

# Extract the best parameters (remove AUC column)
best_params_model <- xgb_model$bestTune

# Log each of the best parameters in MLflow
for (param in names(best_params_model)) {
  mlflow_log_param(param, best_params_model[[param]])
}

# Log the model type as a parameter
mlflow_log_param("model_type", "scm-xgboost-classification")

damage_fit_class_full <- train(
          damage_binary_2 ~ wind_max_pred +
            rain_total_pred +
            roof_strong_wall_strong +
            roof_strong_wall_light +
            roof_strong_wall_salv +
            roof_light_wall_strong +
            roof_light_wall_light +
            roof_light_wall_salv +
            roof_salv_wall_strong +
            roof_salv_wall_light +
            roof_salv_wall_salv +
            #ls_risk_pct + Not local variables
            #ss_risk_pct +
            wind_blue_ss +
            wind_yellow_ss +
            wind_orange_ss +
            wind_red_ss +
            rain_blue_ss +
            rain_yellow_ss +
            rain_orange_ss +
            rain_red_ss +
           track_min_dist, # Confounder adjustment
          data = df_base_train2, # USE TRAINING AND VALIDATION SETS COMBINED
          method = "xgbTree", # XGBoost method
          trControl = trainControl(method = "none"),  # No automatic validation
          tuneGrid = best_params_model # USE BEST PARAMETER
        )


# summarize results
conf_matrix <- confusionMatrix(y_pred,
                     df_base_train2$damage_binary_2,
                     positive = "Damage_above_10"
                     )

# accuracy
accuracy  <- conf_matrix$overall['Accuracy']

# Positive class = 1, precision, recall, and F1
# Extract precision, recall, and F1 score
precision <- conf_matrix$byClass['Precision']
recall <- conf_matrix$byClass['Recall']
f1_score <- conf_matrix$byClass['F1']


# Log parameters and metrics
# mlflow_log_param("model_type", "scm-xgboost-classification")
mlflow_log_metric("accuracy", accuracy)
mlflow_log_metric("F1", f1_score)
mlflow_log_metric("Precision", precision)
mlflow_log_metric("Recall", recall)


# Save model
#saveRDS(model, file = file.path(path_2_folder, "spam_clas_model.rds"))

# End MLflow run
mlflow_end_run()

In [0]:
# Recipe outputs
ass_XGBOOST_classifier <- dkuManagedFolderPath("fZ8zhmA4")

file.path  <- 
saveRDS(, file = )